# Primer: environments, kernels and Python essentials

**Everyone** should read **Environments and kernels** before Session 1, even if Python syntax is familiar. Work through the rest if you are newer to Python or moving from MATLAB, IDL, R or Mathematica *(hover over dotted-underlined terms such as kernel, virtual environment or lockfile for a quick definition)*.

Run cells top to bottom with **Shift+Enter** (in JupyterLab or VS Code), predict each output first, and check finished work with **Kernel → Restart Kernel and Run All Cells**.

## Environments and kernels: pip, conda and uv

- **Why use a virtual environment and dependency solver:** Libraries need compatible versions (e.g. `astropy` and `scipy` sharing `numpy`), which a **dependency solver** checks together:
  - Plain `pip install pkg` has no default lockfile, so later installs can silently upgrade `numpy` and break older scripts.
  - **`uv`** *(used here)* solves `pyproject.toml` from PyPI into a lockfile (`uv.lock`, synced via `uv sync --frozen --extra teaching`).
  - **`conda` / `mamba`** solves Python plus compiled C, Fortran, and GPU libraries (`FFTW`, `CFITSIO`, MPI/CUDA) from conda-forge.

![Diagram: Python environments, package managers and Jupyter kernels](../book/diagrams/python-environments.svg)

**One virtual environment per project, registered as a named kernel** ([source](https://github.com/xangma/jupyter_data_languages/blob/codex/phd-python-2026/book/diagrams/python-environments.dot)).

- **`Kernel != terminal` (and hidden notebook state):**
  - A notebook **kernel** runs one `sys.executable` (e.g. `.venv/bin/python`); new shells do **not** activate that virtual environment automatically — run `source .venv/bin/activate` (`conda activate <env>` or `uv run`).
  - Never `pip install` into system Python or Conda `(base)`. Use `uv add` / `uv sync` in `uv` projects (`python -m pip` or `%pip` in standard `venv`s), and install conda-forge packages *before* `pip`.
  - Out-of-order execution leaves **hidden state** in memory — verify via **Restart Kernel and Run All Cells**.

:::{dropdown} Reference table: pip + venv vs uv vs conda / mamba
| Tool | What its dependency solver manages | Spec & lockfile | Best research fit |
| --- | --- | --- | --- |
| **`pip` + `venv`** | PyPI packages against current env (no default lockfile) | `pyproject.toml`, `requirements.txt` | Simple pure-Python projects |
| **`uv`** *(used here)* | Python + `.venv` + fast full-project PyPI solver | `pyproject.toml` + `uv.lock` | Fast, reproducible project environments |
| **`conda` / `mamba`** | Python + compiled C, Fortran, and CUDA libraries | `environment.yml` (`conda-lock.yml`) | Binary HPC/astro stacks (`FFTW`, `CFITSIO`, MPI/CUDA) |
:::

## Names, containers and imports

- **Imports and help:** `import numpy as np` loads NumPy under the short alias `np`; call `help(fn)` to inspect any unfamiliar function.
- **Containers:** a list `[1.0, 3.0]` is mutable (editable in place), a tuple `(10.0, 20.0)` is an immutable record supporting unpacking (`lo, hi = (10.0, 20.0)`), and a dictionary `{'unit': 'km/s'}` maps named keys to values.
- **0-based indexing and slices:** index `0` is first, `-1` is last, `start:stop` includes `start` while **excluding `stop`**, and `start:stop:step` strides or reverses (`[::2]`, `[::-1]`).
- **Formatted text:** f-strings insert formatted numbers cleanly into text (`f"{val:.2f}"`, `f"{val:.2e}"`).

In [ ]:
import sys
import numpy as np

print('Interpreter:', sys.executable)
measurements = [1.0, 3.0, 5.0, 7.0]
lo, hi = (10.0, 20.0)  # tuple unpacking (immutable record)
observation = {'object_id': 'mock-01', 'unit': 'km/s'}
val = 1234.567
print('First/last:', measurements[0], measurements[-1], '| bounds:', lo, hi, '| unit:', observation['unit'])
print('Slices [:2], [::2], [::-1]:', measurements[:2], measurements[::2], measurements[::-1])
print(f'Formatted f-strings: {val:.2f} | {val:.2e}')

## Lists vs NumPy arrays and masks

- **Whole-array maths:** `[1, 2] + [3, 4]` joins Python lists, whereas `np.array(...)` builds a typed NumPy array so arithmetic acts on every element at once.
- **Missing entries and masks:** NaN (`np.nan`) marks missing or undefined values; test with `np.isfinite(...)` and combine conditions into a boolean mask using `&` or `|` **with parentheses around each comparison**.
- **Attributes, methods and 2D axes:** stored **attributes** (`.shape`, `.dtype`) take no `()`, whereas **methods** (`.mean()`, `.sum()`) compute a result with `()` — on a 2D `(rows, cols)` array, `axis=0` collapses down rows and `axis=1` collapses across columns.

![Diagram: 0-based slice bounds and 2D NumPy axis reductions](../book/diagrams/array-axes-slicing.svg)

**Slice bounds and 2D axes:** slice `stop` indices are exclusive, and `axis` names the dimension being collapsed ([source](https://github.com/xangma/jupyter_data_languages/blob/codex/phd-python-2026/book/diagrams/array-axes-slicing.dot)).

In [ ]:
values = np.array([1.0, 3.0, np.nan, 7.0])
keep = np.isfinite(values) & (values > 2.0)
print('1D attributes -> shape:', values.shape, '| dtype:', values.dtype)
print('1D kept:', values[keep], '| mean():', values[keep].mean())
assert values[keep].tolist() == [3.0, 7.0]

grid_2d = np.array([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])  # shape (2, 3)
print('2D [0, 1]:', grid_2d[0, 1], '| col 0:', grid_2d[:, 0])
print('axis=0 mean:', grid_2d.mean(axis=0), '| axis=1 sum:', grid_2d.sum(axis=1))
assert grid_2d.mean(axis=0).shape == (3,) and grid_2d.sum(axis=1).shape == (2,)

## Loops, vectorised arrays and functions

- **Loops vs vectorised arrays:** `for` loops use 4-space indentation (`enumerate` gives `(index, item)`, `zip` pairs sequences, and `[f"band-{b}" for b in ...]` builds short label lists), whereas numerical maths belongs in vectorised NumPy expressions.
- **Functions and file safety:** wrap reusable maths in `def` functions with a short docstring (`return` hands values back to code and tests, whereas `print` only displays text), and open files with `with` so they close automatically.
- **Explicit plots:** create Matplotlib figures with `fig, ax = plt.subplots(...)` so labels and curves target `ax` directly.

In [ ]:
def square(value):
    """Return the elementwise square of a scalar or NumPy array."""
    return value ** 2

bands = [f"band-{b}" for b in ('g', 'r', 'i')]
for idx, (band, item) in enumerate(zip(bands, [1.0, 2.0, 3.0])):
    print(f"{idx}: {band} -> {square(item)}")

grid = np.array([1.0, 2.0, 3.0, 4.0])
print('Vectorised array output:', square(grid))
assert square(4.0) == 16.0

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(5.5, 2.6))
ax.plot(grid, square(grid), 'o-')
ax.set(xlabel='Grid input x', ylabel='square(x)', title='First Figure + Axes plot')
fig.tight_layout()
plt.show()

## Read a traceback

Read a Python traceback **from the bottom line up**: check the final error type and message first, then look up to the `---->` arrow marking the failing line in your code.

```text
Traceback (most recent call last):
  Cell In[4], line 2
    1 fluxes = np.array([10.0, 20.0, 30.0])
----> 2 calibrated = fluxes * np.array([0.9, 1.1])
ValueError: operands could not be broadcast together with shapes (3,) (2,)
```

`ValueError` shows that 1D shapes `(3,)` and `(2,)` do not match on line 2. Check `.shape` on both arrays and supply matching lengths:

In [ ]:
fluxes = np.array([10.0, 20.0, 30.0])
bad_gains = np.array([0.9, 1.1])
try:
    fluxes * bad_gains
except ValueError as exc:
    print(f'{type(exc).__name__}: {exc}')

gains = np.array([0.9, 1.0, 1.1])
assert fluxes.shape == gains.shape == (3,)
calibrated = fluxes * gains
print('Calibrated:', calibrated)

## Views vs `.copy()` in memory

Unlike MATLAB, R or Mathematica — which copy on write — a NumPy slice is a **view** into the same memory, so modifying a slice also changes the original array. Call `.copy()` when you need an independent array.

![Diagram: A NumPy slice shares memory; .copy() allocates an independent array](../book/diagrams/array-views.svg)

**Shared buffer vs independent copy:** slicing aliases `original`; `.copy()` allocates separate memory ([source](https://github.com/xangma/jupyter_data_languages/blob/codex/phd-python-2026/book/diagrams/array-views.dot)).

In [ ]:
original = np.arange(5)
view = original[:2]
view[0] = 99
print('After view[0] = 99     -> original:', original)
assert original[0] == 99

separate = original[:2].copy()
separate[0] = -5
print('After separate[0] = -5 -> original:', original, '| separate:', separate)
assert original[0] == 99

## Your turn

1. Write `positive_finite(values)` returning a 1D array of finite entries strictly greater than zero (`> 0`).
2. Predict its output on `[-2, 0, 1, np.nan, np.inf, 3]`, then verify with `assert` *(hint: begin with `values = np.asarray(values, dtype=float)`).*
3. Explain why replacing missing values with `0.0` corrupts a mean or model fit.

**Ready for Session 1:** check that you can explain lockfile reproducibility, `Kernel != terminal`, `.shape` vs `.mean()`, a parenthesised `&` boolean mask, and `return`. Compare with [solutions](../book/solutions.md) once your assertion passes.

In [ ]:
# Define positive_finite(values) and check it with an assert